# Core Regressions: Anti-Labor Intensity & Coverage Volume

Tests whether newspaper owners with railroad financial ties produced more anti-labor coverage.
Runs the main regression specifications and coverage volume regressions.

In [10]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/newspapers.db'
OE_PATH = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

# Load intermediate data from 01_data_preparation
df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')


oe = pd.read_csv(OE_PATH)

print(f'Loaded analysis sample: {len(df)} newspaper-years')

Loaded analysis sample: 293 newspaper-years


## Regression

In [11]:
# Prepare data
df['year_str'] = df['year'].astype(str)

# Build circulation lookup from master.csv (nearest Ayer directory year)
import numpy as np
master = pd.read_csv('../data/master.csv', low_memory=False)
circ_cols = [c for c in master.columns if 'circulation' in c.lower()]
circ_long = []
for col in circ_cols:
    yr = int(col.split()[0])
    tmp = master[['issn', col]].dropna(subset=['issn', col]).copy()
    tmp = tmp.rename(columns={col: 'circulation'})
    tmp['circ_year'] = yr
    circ_long.append(tmp)
circ_long = pd.concat(circ_long, ignore_index=True)
circ_long['circulation'] = pd.to_numeric(circ_long['circulation'], errors='coerce')
circ_long = circ_long.dropna(subset=['circulation'])
circ_long = circ_long[circ_long['circulation'] > 0]

circ_lookup = {}
for _, row in circ_long.iterrows():
    circ_lookup.setdefault(row['issn'], []).append((row['circ_year'], row['circulation']))

def nearest_circ(issn, year):
    entries = circ_lookup.get(issn)
    if not entries:
        return None
    return min(entries, key=lambda x: abs(x[0] - year))[1]

df['circulation'] = df.apply(lambda r: nearest_circ(r['issn'], r['year']), axis=1)
df['log_circulation'] = np.log(pd.to_numeric(df['circulation'], errors='coerce'))

print(f"Circulation available for {df['log_circulation'].notna().sum()} / {len(df)} obs")

# Sanity check: 10 newspapers with county population
town_lookup = master.set_index('issn')['town'].to_dict()
check = (df.drop_duplicates('issn')[['issn', 'state', 'county_pop', 'log_county_pop']]
         .assign(town=lambda x: x['issn'].map(town_lookup))
         .dropna(subset=['log_county_pop'])
         .head(10)[['town', 'state', 'county_pop', 'log_county_pop']])
print("\nCounty population sanity check:")
print(check.to_string(index=False))

Circulation available for 173 / 293 obs

County population sanity check:
          town                state  county_pop  log_county_pop
       Belfast                MAINE     34316.1       10.443370
      New York             NEW YORK    942292.0       13.756070
    Saint Paul                  NaN     41329.0       10.629320
    FORT WORTH                TEXAS     41142.0       10.624785
    Alexandria             VIRGINIA     16755.0        9.726452
    Sacramento           CALIFORNIA     40339.0       10.605074
    WASHINGTON DISTRICT OF COLUMBIA    198731.2       12.199708
Salt Lake City                  NaN     45217.0       10.719228
     Lancaster         PENNSYLVANIA    139447.0       11.845440
        Canton                 OHIO     53660.3       10.890429


### Cross-Sectional Regression (newspaper-level)

Collapses all years into a single observation per newspaper: total anti-labor / total labor articles across the owner's full tenure. N = number of newspapers, not newspaper-years.

In [12]:
# Collapse to one observation per newspaper (sum articles across all years)
df_xs = (
    df.groupby('issn')
    .agg(anti_labor=('anti_labor', 'sum'),
         total_labor=('total_labor', 'sum'),
         railroad_interest=('railroad_interest', 'max'),
         person_republican=('person_republican', 'max'),
         log_circulation=('log_circulation', 'mean'),
         log_county_pop=('log_county_pop', 'mean'))
    .reset_index()
)
df_xs['anti_labor_intensity'] = df_xs['anti_labor'] / df_xs['total_labor']

# OLS (HC3)
m0a = smf.ols('anti_labor_intensity ~ railroad_interest', data=df_xs).fit(cov_type='HC3')

df_xs_pol = df_xs.dropna(subset=['person_republican'])
m0b = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican',
              data=df_xs_pol).fit(cov_type='HC3')

df_xs_pop = df_xs.dropna(subset=['person_republican', 'log_county_pop'])
m0d = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              data=df_xs_pop).fit(cov_type='HC3')

from statsmodels.iolib.summary2 import summary_col
print("Cross-sectional regressions — OLS (HC3 SEs)")
print()
print(summary_col(
    [m0a, m0b, m0d],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

# WLS (weighted by total_labor, HC3)
print("\n\nCross-sectional regressions — WLS weighted by total_labor (HC3 SEs)")
print()
wxa = smf.wls('anti_labor_intensity ~ railroad_interest',
              weights=df_xs['total_labor'], data=df_xs).fit(cov_type='HC3')
wxb = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican',
              weights=df_xs_pol['total_labor'], data=df_xs_pol).fit(cov_type='HC3')
wxd = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              weights=df_xs_pop['total_labor'], data=df_xs_pop).fit(cov_type='HC3')

print(summary_col(
    [wxa, wxb, wxd],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

Cross-sectional regressions — OLS (HC3 SEs)


                  (0a) Bivariate (0b) + Party (0c) + Party + Pop
----------------------------------------------------------------
railroad_interest 0.0914**       0.1025**     0.1228**          
                  (0.0370)       (0.0414)     (0.0494)          
person_republican                0.0730*      0.0793*           
                                 (0.0384)     (0.0417)          
log_county_pop                                -0.0497*          
                                              (0.0299)          
Intercept         0.3177***      0.2629***    0.7786**          
                  (0.0197)       (0.0318)     (0.3052)          
R-squared         0.1087         0.1811       0.2988            
R-squared Adj.    0.0915         0.1430       0.2449            
N                 54.0000        46.0000      43.0000           
R²                0.1090         0.1810       0.2990            
Standard errors in parentheses.
* p<.1, ** p

### Panel Regressions (newspaper-year level, OLS, HC3 SEs)

One observation per newspaper-year. Progressively adds year fixed effects, person-level political affiliation, and county population controls.

In [13]:
# Panel OLS regressions (HC3 SEs)
m1 = smf.ols('anti_labor_intensity ~ railroad_interest', data=df).fit(cov_type='HC3')
m2 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df).fit(cov_type='HC3')

df_pol = df.dropna(subset=['person_republican']).copy()
m3 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             data=df_pol).fit(cov_type='HC3')

df_pol_pop = df.dropna(subset=['person_republican', 'log_county_pop']).copy()
m4 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_pol_pop).fit(cov_type='HC3')

from statsmodels.iolib.summary2 import summary_col
print(f"N: full={len(df)} | +party={len(df_pol)} | +party+pop={len(df_pol_pop)}")
print()
print(summary_col(
    [m1, m2, m3, m4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

N: full=293 | +party=256 | +party+pop=250


                  (1) Bivariate (2) Year FE (3) + Party (4) + Party + Pop
-------------------------------------------------------------------------
railroad_interest 0.0678***     0.0629**    0.0602**    0.0674**         
                  (0.0255)      (0.0246)    (0.0269)    (0.0287)         
person_republican                           0.0993***   0.1103***        
                                            (0.0253)    (0.0281)         
log_county_pop                                          -0.0212*         
                                                        (0.0122)         
Intercept         0.2886***     0.1959***   0.1689***   0.3895***        
                  (0.0165)      (0.0618)    (0.0598)    (0.1376)         
C(year)[T.1871]                 -0.0158     -0.0530     -0.0455          
                                (0.0703)    (0.0660)    (0.0679)         
C(year)[T.1872]                 0.1322      0.1028      0.0974      

### Model 4: Editor vs Publisher Split

Runs the main specification (railroad_interest + year FE) separately for newspaper-years where the coded person is an editor vs a publisher.

In [14]:
# Split by role: editors only vs publishers only
df_editors = df[df['has_editor'] == 1].copy()
df_publishers = df[df['has_publisher'] == 1].copy()

print(f"Newspaper-years with editor:    {len(df_editors)}")
print(f"Newspaper-years with publisher: {len(df_publishers)}")

m4_ed = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_editors).fit(cov_type='HC3')
m4_pub = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_publishers).fit(cov_type='HC3')

print()
print(summary_col(
    [m4_ed, m4_pub],
    model_names=['Editors only', 'Publishers only'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'Intercept']
))

Newspaper-years with editor:    254
Newspaper-years with publisher: 226


                  Editors only Publishers only
----------------------------------------------
railroad_interest 0.0498*      0.0678**       
                  (0.0272)     (0.0313)       
Intercept         0.1966***    0.1962***      
                  (0.0674)     (0.0732)       
C(year)[T.1871]   -0.0130      -0.0389        
                  (0.0768)     (0.0848)       
C(year)[T.1872]   0.1853*      0.1504         
                  (0.0989)     (0.1074)       
C(year)[T.1873]   0.0458       0.0285         
                  (0.0808)     (0.0845)       
C(year)[T.1876]   0.0747       0.0951         
                  (0.1101)     (0.1237)       
C(year)[T.1877]   0.3187***    0.2999***      
                  (0.0795)     (0.0842)       
C(year)[T.1878]   0.0090       0.0117         
                  (0.0725)     (0.0782)       
C(year)[T.1879]   0.0969       0.1122         
                  (0.0928)     (0

### Robustness: WLS + Clustered Standard Errors

Re-runs the main specifications using WLS (weighted by `total_labor`) and clustering standard errors at the newspaper level to account for within-newspaper serial correlation.

In [15]:
# WLS weighted by total_labor, SEs clustered by newspaper
cluster_kw = {'groups': df['issn']}
cluster_kw_pol = {'groups': df_pol['issn']}
cluster_kw_pop = {'groups': df_pol_pop['issn']}

w1 = smf.wls('anti_labor_intensity ~ railroad_interest',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w2 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year)',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w3 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             weights=df_pol['total_labor'], data=df_pol).fit(cov_type='cluster', cov_kwds=cluster_kw_pol)
w4 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             weights=df_pol_pop['total_labor'], data=df_pol_pop).fit(cov_type='cluster', cov_kwds=cluster_kw_pop)

print("WLS (weighted by total_labor) + newspaper-clustered SEs")
print()
print(summary_col(
    [w1, w2, w3, w4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year+Party', '(4) +County Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'Clusters': lambda m: len(m.cov_kwds['groups'].unique()),
               'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

WLS (weighted by total_labor) + newspaper-clustered SEs


                  (1) Bivariate (2) Year FE (3) Year+Party (4) +County Pop
--------------------------------------------------------------------------
railroad_interest 0.0615***     0.0471**    0.0555***      0.0505**       
                  (0.0204)      (0.0184)    (0.0188)       (0.0239)       
person_republican                           0.0439**       0.0488*        
                                            (0.0186)       (0.0264)       
log_county_pop                                             0.0035         
                                                           (0.0097)       
Intercept         0.3017***     0.2026***   0.1706***      0.1227         
                  (0.0129)      (0.0343)    (0.0332)       (0.0993)       
C(year)[T.1871]                 0.0528      0.0667*        0.0703*        
                                (0.0349)    (0.0395)       (0.0392)       
C(year)[T.1872]                 0.0618    

---

## 6. Coverage Amount Regression

Does railroad financial interest affect the **volume** of labor coverage?

**Outcome:** `labor_coverage_share` = labor articles / total articles per newspaper-year  
**Treatment:** `railroad_interest` (same as above)

In [16]:
# Compute total articles per newspaper-year for ISSNs in the analysis sample
conn = sqlite3.connect(DB_PATH)

analysis_issns = df['issn'].unique().tolist()
placeholders = ','.join(['?'] * len(analysis_issns))

total_articles = pd.read_sql(f"""
    SELECT issn, year, COUNT(*) as total_articles
    FROM articles
    WHERE issn IN ({placeholders})
    GROUP BY issn, year
""", conn, params=analysis_issns)

conn.close()

# Merge: labor article counts (from 'counts' df) + total articles
coverage = counts[['issn', 'year', 'total_labor']].merge(
    total_articles, on=['issn', 'year'], how='inner'
)
coverage['labor_coverage_share'] = coverage['total_labor'] / coverage['total_articles']

# Merge with treatment
df_cov = coverage.merge(paper_year_rr, on=['issn', 'year'], how='inner')
df_cov['year_str'] = df_cov['year'].astype(str)

print(f"Coverage analysis sample: {len(df_cov)} newspaper-years")
print(f"\nLabor coverage share by treatment group:")
print(df_cov.groupby('railroad_interest')['labor_coverage_share'].describe())

Coverage analysis sample: 293 newspaper-years

Labor coverage share by treatment group:
                   count      mean       std       min       25%       50%  \
railroad_interest                                                            
0                  172.0  0.001881  0.001616  0.000147  0.000704  0.001488   
1                  121.0  0.002046  0.001679  0.000120  0.000818  0.001669   

                        75%       max  
railroad_interest                      
0                  0.002533  0.009834  
1                  0.002755  0.009645  


In [17]:
# Merge person_republican and log_county_pop into coverage df
df_cov = df_cov.merge(
    df[['issn', 'year', 'person_republican', 'log_county_pop']],
    on=['issn', 'year'], how='left'
)

# Coverage regressions
c1 = smf.ols('labor_coverage_share ~ railroad_interest', data=df_cov).fit(cov_type='HC3')
c2 = smf.ols('labor_coverage_share ~ railroad_interest + C(year)', data=df_cov).fit(cov_type='HC3')

df_cov_pol = df_cov.dropna(subset=['person_republican'])
c3 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican',
             data=df_cov_pol).fit(cov_type='HC3')

df_cov_pop = df_cov.dropna(subset=['person_republican', 'log_county_pop'])
c4 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_cov_pop).fit(cov_type='HC3')

print(summary_col(
    [c1, c2, c3, c4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))


                  (1) Bivariate (2) Year FE (3) + Party (4) + Party + Pop
-------------------------------------------------------------------------
railroad_interest 0.0002        0.0001      0.0002      -0.0002          
                  (0.0002)      (0.0002)    (0.0002)    (0.0002)         
person_republican                           -0.0000     -0.0000          
                                            (0.0002)    (0.0002)         
log_county_pop                                          0.0003***        
                                                        (0.0001)         
Intercept         0.0019***     0.0009***   0.0010***   -0.0026***       
                  (0.0001)      (0.0002)    (0.0003)    (0.0008)         
C(year)[T.1871]                 0.0005      0.0005      0.0004           
                                (0.0006)    (0.0008)    (0.0008)         
C(year)[T.1872]                 0.0005      0.0004      0.0005           
                                (0.00